# 🏗️ Multi-Agent Systems with LangGraph Subgraphs

## Welcome to Session 6, Lab 1!

In this lab, you'll learn how to build **multi-agent systems** using LangGraph's powerful **subgraph** architecture. Subgraphs let you compose complex workflows from smaller, self-contained agent pipelines — a critical pattern for building production-grade AI applications.

---

### 🎯 Learning Objectives

By the end of this notebook, you will be able to:

- **Define agent functions** that operate on shared state and pass data between each other
- **Build subgraphs** — self-contained LangGraph pipelines that encapsulate a multi-step workflow
- **Embed subgraphs into parent graphs** — compose larger systems from smaller building blocks
- **Create LLM-powered agent pipelines** where each agent calls an LLM with a specialized prompt
- **Unify multiple subgraphs** into a single orchestrated workflow that chains different capabilities

---

### 🧠 Key Concepts

- **StateGraph**: LangGraph's core abstraction — a directed graph where nodes are functions and edges define execution order
- **Subgraph**: A compiled StateGraph that can be embedded as a single node inside a parent graph
- **Shared State (dict or TypedDict)**: The data structure passed between agents; each agent reads from and writes to this state
- **Entry Point / Finish Point**: Declares where execution begins and ends within a graph
- **Compilation**: Converts a graph definition into an executable object that can be invoked

---

### 📋 Prerequisites

- Python 3.9+
- `langgraph`, `langchain-openai` packages installed
- An OpenAI API key (for the LLM-powered sections)
- Familiarity with basic Python functions and dictionaries

---

### 🗺️ Notebook Structure

| Section | Description |
|---------|-------------|
| **Part 1** | Simple E-Commerce Flow — rule-based agents with a subgraph |
| **Part 2** | LLM-Powered Research Pipeline — ChatOpenAI agents in a subgraph |
| **Part 3** | Unified Parent Graph — combining both subgraphs into one workflow |

Let's get started! 🚀

## ⚡ Part 1: Simple E-Commerce Multi-Agent Flow

In this section, we'll build a **rule-based multi-agent pipeline** that simulates an e-commerce order workflow. Three agents collaborate in sequence:

1. **UserAgent** — Places the order
2. **QuantityAgent** — Checks stock availability
3. **PaymentAgent** — Processes the payment

### 🧠 Key Concepts

- **Shared state as a plain `dict`**: The simplest way to pass data between agents — each agent reads/writes keys in a dictionary
- **Sequential agent flow**: Agents execute in a fixed order defined by graph edges
- **Guard clauses**: Each agent validates preconditions before proceeding (e.g., "is the order placed?")
- **Subgraph compilation**: We'll compile this 3-agent pipeline into a subgraph, then embed it in a parent graph

### 🔧 What You'll Build

```
UserAgent → QuantityAgent → PaymentAgent
```

Let's start by importing the required libraries.

In [ ]:
# 📦 Import core LangGraph components
from langgraph.graph import StateGraph, END  # StateGraph: builds the agent workflow graph; END: special sentinel marking graph termination
from IPython.display import Image              # Image: renders PNG visualizations of the graph inline in Jupyter

print("✅ LangGraph and display utilities imported successfully!")

In [ ]:
# 🤖 Define the three e-commerce agent functions
# Each agent is a simple Python function that takes a shared state dict,
# performs its task, updates the state, and returns the modified state.
# This is the fundamental pattern in LangGraph: nodes are functions that transform state.

def user_agent(state):
    """Agent 1: Simulates a user placing an order.
    Sets 'order_placed' to True in the shared state."""
    print("🛒 UserAgent: Placing the order...")
    state["order_placed"] = True               # Mark the order as placed in shared state
    return state                                # Return updated state for the next agent

def quantity_agent(state):
    """Agent 2: Checks whether the ordered item is in stock.
    Requires 'order_placed' to be True; sets 'quantity_available'."""
    if not state.get("order_placed"):           # Guard clause: ensure the order was placed first
        raise ValueError("Order not placed, try after some time.")
    print("📦 QuantityAgent: Checking available stock...")
    state["quantity_available"] = True          # Set to False to simulate an out-of-stock scenario
    return state                                # Return updated state for the next agent

def payment_agent(state):
    """Agent 3: Processes payment for the order.
    Requires 'quantity_available' to be True; sets 'payment_done'."""
    if not state.get("quantity_available"):     # Guard clause: ensure stock is available before payment
        raise ValueError("Insufficient quantity, can't proceed to payment.")
    print("💳 PaymentAgent: Processing payment...")
    state["payment_done"] = True                # Mark payment as completed
    return state                                # Return final updated state

print("✅ E-commerce agent functions defined: UserAgent, QuantityAgent, PaymentAgent")

### 🔀 Building the E-Commerce Subgraph

Now we'll wire these three agents into a **StateGraph** — LangGraph's directed graph abstraction. The steps are:

1. **Create a StateGraph** with `dict` as the state type (simple key-value store)
2. **Add nodes** — each node is one of our agent functions
3. **Add edges** — define the execution order (UserAgent → QuantityAgent → PaymentAgent)
4. **Set entry/finish points** — declare where the graph starts and ends
5. **Compile** — convert the graph definition into an executable object
6. **Visualize** — render the graph as a diagram to confirm the structure

In [ ]:
# 🔨 Build the e-commerce subgraph
# StateGraph(dict) creates a graph that uses a plain Python dict as shared state.
# This is the simplest state type — great for prototyping. For production, use TypedDict.
subgraph = StateGraph(dict)

# ➕ Add each agent function as a named node in the graph
# The first argument is the node name (string), the second is the function to execute
subgraph.add_node("UserAgent", user_agent)         # Node 1: places the order
subgraph.add_node("QuantityAgent", quantity_agent)  # Node 2: checks stock
subgraph.add_node("PaymentAgent", payment_agent)    # Node 3: processes payment

# 🔗 Define the execution flow using edges (directed connections between nodes)
subgraph.set_entry_point("UserAgent")               # Execution starts at UserAgent
subgraph.add_edge("UserAgent", "QuantityAgent")     # After UserAgent, go to QuantityAgent
subgraph.add_edge("QuantityAgent", "PaymentAgent")  # After QuantityAgent, go to PaymentAgent

# 🏁 Mark PaymentAgent as the final node — execution stops after this node completes
subgraph.set_finish_point("PaymentAgent")

# ⚙️ Compile the graph into an executable object
# Compilation validates the graph structure and creates an invocable workflow
compiled_subgraph = subgraph.compile()

# 🎨 Visualize the compiled graph as a Mermaid diagram rendered to PNG
# This helps confirm the agent flow matches our intended design
png_bytes = compiled_subgraph.get_graph().draw_mermaid_png()
Image(png_bytes)

### 🏗️ Embedding the Subgraph in a Parent Graph

This is the **key architectural pattern** of this lab: taking a compiled subgraph and embedding it as a **single node** inside a parent graph.

**Why does this matter?**
- **Modularity**: Each subgraph is a self-contained unit that can be developed and tested independently
- **Composability**: Parent graphs can combine multiple subgraphs into complex workflows
- **Encapsulation**: The parent graph doesn't need to know the internal structure of its subgraphs
- **Reusability**: The same compiled subgraph can be reused in multiple parent graphs

Think of it like a function call: the parent graph sees "EcommerceFlow" as a single step, but internally it runs three agents in sequence.

In [ ]:
# 🏗️ Create a parent graph and embed the e-commerce subgraph as a single node
parent_graph = StateGraph(dict)

# 🔌 Add the compiled subgraph as a node named "EcommerceFlow"
# LangGraph treats compiled subgraphs just like regular functions —
# the parent graph invokes it as one atomic step, but internally
# it runs UserAgent → QuantityAgent → PaymentAgent in sequence.
parent_graph.add_node("EcommerceFlow", compiled_subgraph)

# Set entry and finish points — this parent graph has just one node
parent_graph.set_entry_point("EcommerceFlow")      # Start at the subgraph
parent_graph.set_finish_point("EcommerceFlow")     # End after the subgraph completes

# ⚙️ Compile the parent graph
compiled_parent_graph = parent_graph.compile()

# 🧪 Initialize the shared state with all flags set to False
# Each agent will flip its corresponding flag to True as it executes
state = {
    "order_placed": False,        # Will be set True by UserAgent
    "quantity_available": False,   # Will be set True by QuantityAgent
    "payment_done": False          # Will be set True by PaymentAgent
}

# 🚀 Execute the parent graph — this triggers the entire e-commerce pipeline
print("=" * 50)
print("🚀 Executing E-Commerce Parent Graph")
print("=" * 50)
result = compiled_parent_graph.invoke(state)

# 📊 Display the final state to confirm all agents executed successfully
print("\n✅ Final State:", result)

---

## 🧠 Part 2: LLM-Powered Multi-Agent Research Pipeline

Now we'll level up from rule-based agents to **LLM-powered agents**. Each agent in this section calls OpenAI's GPT-4o model with a specialized prompt, creating a research pipeline:

1. **ResearchAgent** — Gathers information on a given topic
2. **AnalysisAgent** — Analyzes the research findings for deeper insights
3. **SummaryAgent** — Produces a concise final summary

### 🔑 Why This Matters

- **Specialization**: Each agent has a focused role with a tailored prompt, producing better results than a single monolithic prompt
- **Information flow**: Each agent builds on the output of the previous agent, creating a chain of increasingly refined content
- **TypedDict state**: We upgrade from plain `dict` to a `TypedDict` with type annotations — essential for production code
- **Annotated reducers**: Using `Annotated[list, operator.add]` to define how list fields merge across agents

### 🔧 What You'll Build

```
ResearchAgent (LLM) → AnalysisAgent (LLM) → SummaryAgent (LLM)
```

First, let's set up the OpenAI API key.

In [ ]:
# 🔑 Set up the OpenAI API key
# This helper function checks if the environment variable is already set;
# if not, it securely prompts the user to enter it (input is masked with getpass).
import os, getpass

def _set_env(var: str):
    """Securely set an environment variable if it's not already defined."""
    if not os.environ.get(var):                    # Only prompt if the variable isn't set
        os.environ[var] = getpass.getpass(f"{var}: ")  # Masked input — keeps your key hidden

_set_env("OPENAI_API_KEY")  # Ensure OPENAI_API_KEY is available for ChatOpenAI
print("✅ OpenAI API key is configured!")

In [ ]:
# 📦 Import additional dependencies for the LLM-powered pipeline
from typing import TypedDict, Annotated  # TypedDict: structured state; Annotated: attach metadata to types
import operator                           # operator.add: used as a reducer to merge lists across agents
from langchain_openai import ChatOpenAI   # ChatOpenAI: LangChain wrapper for OpenAI's chat models

print("✅ TypedDict, operator, and ChatOpenAI imported successfully!")

### 📊 Defining the Multi-Agent State Schema

Unlike Part 1 where we used a plain `dict`, here we define a **TypedDict** — a typed dictionary that provides:

- **Type safety**: Each field has an explicit type, catching errors early
- **Documentation**: The schema itself documents what data flows through the pipeline
- **Reducer support**: The `Annotated[list, operator.add]` pattern tells LangGraph how to merge list values when multiple agents write to the same field

> **Key insight**: The `messages` field uses `operator.add` as a reducer, meaning each agent **appends** to the list rather than overwriting it. This preserves the full conversation history across agents.

In [ ]:
# 📐 Define the state structure for the multi-agent research pipeline
# TypedDict gives us a strongly-typed schema that LangGraph uses to
# validate state at each step and apply reducers for merging values.

class MultiAgentState(TypedDict):
    messages: Annotated[list, operator.add]  # Accumulated messages from all agents; operator.add merges lists
    task: str                                 # The research topic/task to investigate
    research_done: bool                       # Flag: has the ResearchAgent completed its work?
    analysis_done: bool                       # Flag: has the AnalysisAgent completed its work?
    final_summary: str                        # The final summary produced by the SummaryAgent

print("✅ MultiAgentState schema defined with 5 fields")

### 🤖 Defining the LLM-Powered Agent Functions

Each agent follows the same pattern:
1. **Read** relevant data from the shared state
2. **Construct a prompt** tailored to the agent's role
3. **Call the LLM** via `llm.invoke(prompt)`
4. **Append the result** to the `messages` list in state
5. **Update a status flag** and return the modified state

This creates an **information pipeline** where each agent's output becomes the next agent's input.

In [ ]:
# 🧠 Initialize the LLM (Large Language Model)
# Using GPT-4o with temperature=0.7 for a balance of creativity and coherence.
# Temperature controls randomness: 0.0 = deterministic, 1.0 = highly creative.
llm = ChatOpenAI(model="gpt-4o", temperature=0.7)

# ─────────────────────────────────────────────────────────────────────
# 🔬 Agent 1: Research Agent
# Role: Takes the task description and asks the LLM to research the topic.
# Output: Appends research findings to the messages list.
# ─────────────────────────────────────────────────────────────────────
def research_agent_llm(state: MultiAgentState) -> MultiAgentState:
    print("🔬 ResearchAgent: Gathering information...")
    # Build a research-focused prompt using the task from state
    prompt = f"Research the following topic and provide key insights: {state['task']}"
    # Call the LLM — returns an AIMessage object with a .content attribute
    response = llm.invoke(prompt)
    # Append the research output to the shared messages list
    # Each message is a dict with the agent name and the LLM's response content
    state["messages"].append({"agent": "research", "content": response.content})
    state["research_done"] = True  # Signal that research is complete
    return state

# ─────────────────────────────────────────────────────────────────────
# 📈 Agent 2: Analysis Agent
# Role: Reads the research findings and asks the LLM to analyze them.
# Input: The last message in the messages list (from ResearchAgent).
# Output: Appends analysis results to the messages list.
# ─────────────────────────────────────────────────────────────────────
def analysis_agent_llm(state: MultiAgentState) -> MultiAgentState:
    print("📈 AnalysisAgent: Analyzing data...")
    # Retrieve the most recent message content (the research findings)
    research_content = state["messages"][-1]["content"]
    # Build an analysis-focused prompt that references the research output
    prompt = f"Analyze the following research findings and provide deeper insights: {research_content}"
    response = llm.invoke(prompt)
    # Append analysis results to the messages list
    state["messages"].append({"agent": "analysis", "content": response.content})
    state["analysis_done"] = True  # Signal that analysis is complete
    return state

# ─────────────────────────────────────────────────────────────────────
# 📝 Agent 3: Summary Agent
# Role: Reads ALL accumulated messages and produces a concise summary.
# Input: The full messages list (research + analysis).
# Output: Appends summary to messages AND writes it to final_summary.
# ─────────────────────────────────────────────────────────────────────
def summary_agent_llm(state: MultiAgentState) -> MultiAgentState:
    print("📝 SummaryAgent: Creating final summary...")
    # Concatenate ALL message contents into a single string for comprehensive summarization
    all_content = "\n".join([msg["content"] for msg in state["messages"]])
    # Build a summary-focused prompt with the full research and analysis content
    prompt = f"Create a concise summary of the following research and analysis: {all_content}"
    response = llm.invoke(prompt)
    # Append the summary to messages AND store it in the dedicated final_summary field
    state["messages"].append({"agent": "summary", "content": response.content})
    state["final_summary"] = response.content  # Store final output for easy access
    return state

print("✅ Three LLM-powered agents defined: ResearchAgent, AnalysisAgent, SummaryAgent")

### 🔀 Building the Research Pipeline Subgraph

Now we wire the three LLM agents into a subgraph — the same pattern as Part 1, but with `MultiAgentState` (TypedDict) instead of a plain `dict`. The flow is:

```
Research → Analysis → Summary
```

After compilation, we'll visualize the graph to confirm the structure.

In [ ]:
# 🔨 Build the multi-agent research subgraph
# This time we use MultiAgentState (TypedDict) instead of dict —
# LangGraph will validate state fields and apply the operator.add reducer on 'messages'.
multi_agent_subgraph = StateGraph(MultiAgentState)

# ➕ Add each LLM-powered agent as a named node
multi_agent_subgraph.add_node("Research", research_agent_llm)    # Node 1: researches the topic
multi_agent_subgraph.add_node("Analysis", analysis_agent_llm)    # Node 2: analyzes research findings
multi_agent_subgraph.add_node("Summary", summary_agent_llm)      # Node 3: creates final summary

# 🔗 Define the sequential execution flow
multi_agent_subgraph.set_entry_point("Research")                  # Start with research
multi_agent_subgraph.add_edge("Research", "Analysis")             # Research feeds into analysis
multi_agent_subgraph.add_edge("Analysis", "Summary")              # Analysis feeds into summary
multi_agent_subgraph.set_finish_point("Summary")                  # End after summary is produced

# ⚙️ Compile the subgraph into an executable pipeline
compiled_multi_agent = multi_agent_subgraph.compile()

# 🎨 Visualize the research pipeline graph
png_bytes = compiled_multi_agent.get_graph().draw_mermaid_png()
Image(png_bytes)

### 🧪 Testing the Research Pipeline

Let's test the subgraph by giving it a research topic. The pipeline will:
1. **ResearchAgent** calls GPT-4o to research the topic
2. **AnalysisAgent** reads the research output and provides deeper analysis
3. **SummaryAgent** reads everything and produces a concise summary

Watch the print statements to see each agent activate in sequence!

In [ ]:
# 🧪 Test the research pipeline subgraph
# Initialize state with our research task and empty/default values for all fields
test_state = {
    "messages": [],                                                # Empty list — agents will append their outputs here
    "task": "Explain the benefits of multi-agent systems in AI",   # The research topic for our pipeline
    "research_done": False,                                        # Will be flipped to True by ResearchAgent
    "analysis_done": False,                                        # Will be flipped to True by AnalysisAgent
    "final_summary": ""                                            # Will be populated by SummaryAgent
}

# 🚀 Execute the research pipeline — all three agents run in sequence
print("=" * 60)
print("🚀 Executing Research Pipeline: Research → Analysis → Summary")
print("=" * 60)
result_multi_agent = compiled_multi_agent.invoke(test_state)

# 📊 Display the final summary produced by the SummaryAgent
print("\n" + "=" * 60)
print("📝 === Final Summary ===")
print("=" * 60)
print(result_multi_agent["final_summary"])

### 🏗️ Wrapping the Research Pipeline in a Parent Graph

Just like Part 1, we now embed the compiled research subgraph into a parent graph. This demonstrates the same **composition pattern**: the parent graph sees the entire 3-agent pipeline as a single node called "MultiAgentFlow".

In [ ]:
# 🏗️ Create a parent graph wrapping the multi-agent research pipeline
parent_multi_agent_graph = StateGraph(MultiAgentState)

# 🔌 Embed the compiled research subgraph as a single node
# From the parent's perspective, "MultiAgentFlow" is one step —
# but internally it runs Research → Analysis → Summary
parent_multi_agent_graph.add_node("MultiAgentFlow", compiled_multi_agent)
parent_multi_agent_graph.set_entry_point("MultiAgentFlow")    # Start at the subgraph
parent_multi_agent_graph.set_finish_point("MultiAgentFlow")   # End after the subgraph completes

# ⚙️ Compile the parent graph
compiled_parent_multi_agent = parent_multi_agent_graph.compile()

# 🎨 Visualize — notice the parent graph shows just one node,
# hiding the 3-agent pipeline inside
png_bytes = compiled_parent_multi_agent.get_graph().draw_mermaid_png()
Image(png_bytes)

---

## 🔀 Part 3: Unified Parent Graph — Combining Both Subgraphs

This is where everything comes together! We'll create a **unified parent graph** that chains both subgraphs into a single end-to-end workflow:

```
ProductResearch (LLM subgraph) → PlaceOrder (rule-based subgraph)
```

### 🧠 Key Concepts

- **Unified state**: We define a `UnifiedState` TypedDict that contains fields for BOTH subgraphs — the e-commerce fields AND the research pipeline fields
- **Cross-subgraph composition**: Two independently-built subgraphs are combined in a single parent graph
- **Sequential orchestration**: The parent graph defines the order — research first, then place the order
- **Real-world pattern**: This mirrors real applications where you might research a product with AI agents, then trigger a transactional workflow

### 🎯 Why This Matters

This pattern is the foundation of **production multi-agent systems**:
- Each team can develop their subgraph independently
- Subgraphs can be tested in isolation (as we did in Parts 1 and 2)
- The parent graph orchestrates the overall business workflow
- Adding new capabilities = adding new subgraph nodes

In [ ]:
# 🔀 Create a unified parent graph that combines BOTH subgraphs
from typing import Any

# ─────────────────────────────────────────────────────────────────────
# 📐 Define a UnifiedState that carries fields for BOTH workflows
# This is the "superset" state — it includes everything both subgraphs need.
# Each subgraph only reads/writes the fields it cares about.
# ─────────────────────────────────────────────────────────────────────
class UnifiedState(TypedDict):
    # 🛒 E-commerce fields (used by the PlaceOrder subgraph)
    order_placed: bool                        # Tracks whether the order has been placed
    quantity_available: bool                   # Tracks whether stock is available
    payment_done: bool                        # Tracks whether payment was processed
    # 🔬 Multi-agent research fields (used by the ProductResearch subgraph)
    messages: Annotated[list, operator.add]    # Accumulated agent messages (with reducer)
    task: str                                  # The research task/topic
    research_done: bool                        # Flag: research complete?
    analysis_done: bool                        # Flag: analysis complete?
    final_summary: str                         # The final research summary
    # 📦 Product information (shared context)
    product_name: str                          # Name of the product being researched/ordered
    product_recommendation: str                # AI recommendation for the product

# ─────────────────────────────────────────────────────────────────────
# 🏗️ Build the unified parent graph
# ─────────────────────────────────────────────────────────────────────
unified_parent_graph = StateGraph(UnifiedState)

# 🔌 Add BOTH compiled subgraphs as nodes in the parent graph
# "ProductResearch" — the LLM-powered research pipeline (Research → Analysis → Summary)
# "PlaceOrder"      — the rule-based e-commerce flow (UserAgent → QuantityAgent → PaymentAgent)
unified_parent_graph.add_node("ProductResearch", compiled_multi_agent)
unified_parent_graph.add_node("PlaceOrder", compiled_subgraph)

# 🔗 Define the workflow: research the product FIRST, then place the order
unified_parent_graph.set_entry_point("ProductResearch")           # Start with product research
unified_parent_graph.add_edge("ProductResearch", "PlaceOrder")    # After research, proceed to ordering
unified_parent_graph.add_edge("PlaceOrder", END)                  # After ordering, workflow is complete

# ⚙️ Compile the unified parent graph
compiled_unified_parent = unified_parent_graph.compile()

# 🎨 Visualize the unified workflow — should show: ProductResearch → PlaceOrder → END
png_bytes = compiled_unified_parent.get_graph().draw_mermaid_png()
Image(png_bytes)

### 🚀 Running the Full End-to-End Workflow

Now let's execute the complete unified workflow. This single `invoke()` call will:

1. **ProductResearch subgraph** runs first:
   - ResearchAgent gathers info about wireless noise-cancelling headphones
   - AnalysisAgent provides deeper insights on the research
   - SummaryAgent produces a concise summary
2. **PlaceOrder subgraph** runs second:
   - UserAgent places the order
   - QuantityAgent checks stock availability
   - PaymentAgent processes the payment

All six agents execute in sequence, passing state through the unified pipeline!

In [ ]:
# 🚀 Test the unified parent graph: Research a product, then place an order

# 📦 Define the product to research and order
product_to_buy = "wireless noise-cancelling headphones"

# 🧪 Initialize the unified state with ALL fields required by both subgraphs
# This is the "superset" state — both subgraphs will read/write their respective fields
unified_state = {
    # 🛒 E-commerce fields — all start as False, will be flipped by each agent
    "order_placed": False,
    "quantity_available": False,
    "payment_done": False,
    # 🔬 Multi-agent research fields — empty/default values to start
    "messages": [],                # Will accumulate outputs from Research, Analysis, Summary agents
    "task": f"Research {product_to_buy} and provide key features, benefits, and recommendations for purchase",
    "research_done": False,
    "analysis_done": False,
    "final_summary": "",
    # 📦 Product information — shared across both subgraphs
    "product_name": product_to_buy,
    "product_recommendation": ""
}

# 🚀 Execute the unified workflow — this triggers ALL 6 agents in sequence:
# ResearchAgent → AnalysisAgent → SummaryAgent → UserAgent → QuantityAgent → PaymentAgent
print("=" * 60)
print("🚀 Executing Product Research → Order Placement Workflow")
print("=" * 60)
print(f"\n📦 Product to research: {product_to_buy}\n")

result_unified = compiled_unified_parent.invoke(unified_state)

# ─────────────────────────────────────────────────────────────────────
# 📊 Display the results from both subgraphs
# ─────────────────────────────────────────────────────────────────────
print("\n" + "=" * 60)
print("🔬 === PRODUCT RESEARCH RESULTS ===")
print("=" * 60)
if result_unified.get('final_summary'):
    print(result_unified['final_summary'])

print("\n" + "=" * 60)
print("🛒 === ORDER PLACEMENT RESULTS ===")
print("=" * 60)
print(f"📦 Product: {result_unified['product_name']}")
print(f"🛒 Order Placed: {result_unified['order_placed']}")
print(f"📦 Quantity Available: {result_unified['quantity_available']}")
print(f"💳 Payment Done: {result_unified['payment_done']}")

# ✅ Final status check
if result_unified['payment_done']:
    print("\n✅ Order successfully completed!")
else:
    print("\n❌ Order failed")

---

## 🎓 Summary: What You've Learned

Congratulations! You've completed the Multi-Agent Systems lab. Here's a recap of the key concepts and skills you've practiced:

### 📋 Concepts Covered

| Concept | Description |
|---------|-------------|
| **StateGraph** | LangGraph's core abstraction for building agent workflows as directed graphs |
| **Agent Functions** | Python functions that read from and write to shared state |
| **Subgraphs** | Self-contained compiled graphs that can be embedded as nodes in parent graphs |
| **Parent Graphs** | Higher-level graphs that orchestrate subgraphs into complex workflows |
| **TypedDict State** | Structured state schemas with type annotations for production-quality code |
| **Annotated Reducers** | Using `Annotated[list, operator.add]` to control how state fields merge |
| **Unified State** | A superset state that carries fields for multiple subgraphs |

### 🏗️ Architecture Patterns

1. **Simple Subgraph**: Define agents → wire into graph → compile → invoke
2. **Subgraph in Parent**: Compile subgraph → add as node in parent graph → compose larger workflows
3. **Multi-Subgraph Composition**: Multiple independently-built subgraphs chained in a single parent graph

### 🚀 Next Steps

- **Add conditional routing**: Use `add_conditional_edges()` to branch based on state values (e.g., skip payment if out of stock)
- **Add human-in-the-loop**: Interrupt the graph for human approval before critical steps
- **Add error handling**: Implement retry logic and fallback paths for failed LLM calls
- **Scale horizontally**: Add more specialized agents (e.g., ReviewAgent, ShippingAgent, NotificationAgent)
- **Explore Supervisor patterns**: Use a supervisor agent to dynamically route between worker agents

### 💡 Key Takeaway

> The **subgraph composition pattern** is the foundation of scalable multi-agent systems. By building small, testable subgraphs and composing them in parent graphs, you can create complex AI workflows that remain modular, maintainable, and easy to extend.